In [1]:
import sys

sys.path.append("..")

import os
import random
import uuid
from datetime import datetime, timedelta

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)
print("Engine ready")

Engine ready


In [2]:
SCHEMA_SQL = """
DROP TABLE IF EXISTS orders CASCADE;
DROP TABLE IF EXISTS products CASCADE;

CREATE TABLE products (
    id VARCHAR(50) PRIMARY KEY,
    title VARCHAR(255) NOT NULL,
    category VARCHAR(100),
    price NUMERIC(10, 2),
    stock_count INTEGER NOT NULL DEFAULT 0,
    aisle VARCHAR(50)
);

CREATE TABLE orders (
    order_id VARCHAR(50) PRIMARY KEY,
    product_id VARCHAR(50) REFERENCES products(id),
    quantity INTEGER NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'placed',  -- placed, preparing, out_for_delivery, delivered, cancelled
    created_at TIMESTAMP NOT NULL DEFAULT NOW()
);
"""

In [3]:
with engine.begin() as conn:
    conn.execute(text(SCHEMA_SQL))
print("Tables created: products, orders")

Tables created: products, orders


In [4]:
product_rows = [
    (
        "product_001",
        "Cadbury Dairy Milk Chocolate Bar",
        "confectionery",
        2.99,
        "shelf 7",
    ),
    ("product_002", "FreshMart Whole Wheat Bread", "bakery", 3.49, "shelf 3"),
    ("product_003", "Artisan Sourdough Loaf", "bakery", 5.99, "shelf 3"),
    ("product_004", "FreshMart 2% Milk", "dairy", 2.79, "shelf 12"),
    ("product_005", "Sparkling Water Variety Pack", "beverages", 6.99, "shelf 9"),
    ("product_006", "Kettle-Cooked Potato Chips", "snacks", 3.29, "shelf 5"),
    ("product_007", "Organic Bananas", "produce", 0.69, "entrance"),
    ("product_008", "Recycled Paper Towels", "household", 8.49, "shelf 14"),
    ("product_009", "Cadbury Roses Chocolate Tin", "confectionery", 9.99, "shelf 7"),
    ("product_010", "Frozen Mixed Vegetables", "frozen", 2.49, "shelf 18"),
    ("product_011", "Wireless Bluetooth Earbuds", "electronics", 1999.00, "aisle 22"),
    ("product_012", "Smartphone 128GB Storage", "electronics", 18499.00, "aisle 21"),
    ("product_013", "Smart LED TV 43-inch", "electronics", 24990.00, "aisle 25"),
    ("product_014", "Portable Power Bank 20000mAh", "electronics", 1499.00, "aisle 22"),
    (
        "product_015",
        "Non-Stick Cookware Set (3 pcs)",
        "home_kitchen",
        1299.00,
        "shelf 16",
    ),
    ("product_016", "Electric Kettle 1.5L", "home_kitchen", 899.00, "shelf 17"),
    ("product_017", "Mixer Grinder 750W", "home_kitchen", 2799.00, "shelf 17"),
    ("product_018", "Men's Cotton Casual Shirt", "fashion", 799.00, "aisle 30"),
    ("product_019", "Women's Running Shoes", "fashion", 1899.00, "aisle 31"),
    ("product_020", "Herbal Shampoo 340ml", "personal_care", 249.00, "shelf 19"),
    ("product_021", "Electric Trimmer", "personal_care", 899.00, "shelf 20"),
    ("product_022", "Toned Milk 1L Pouch", "dairy", 58.00, "shelf 12"),
    ("product_023", "Curd/Yogurt 400g Cup", "dairy", 45.00, "shelf 12"),
    ("product_024", "Processed Cheese Slices", "dairy", 135.00, "shelf 13"),
    ("product_025", "Paneer 200g Pack", "dairy", 85.00, "shelf 13"),
    ("product_026", "Basmati Rice 5kg", "grocery", 649.00, "shelf 2"),
    ("product_027", "Toor Dal (Split Pigeon Peas) 1kg", "grocery", 165.00, "shelf 2"),
    ("product_028", "Refined Sunflower Oil 1L", "grocery", 139.00, "shelf 1"),
    ("product_029", "Dishwash Liquid Gel 500ml", "household", 149.00, "shelf 14"),
    ("product_030", "Laundry Detergent Powder 1kg", "household", 185.00, "shelf 15"),
]

random.seed(42)

with engine.begin() as conn:
    for product_id, title, category, price, aisle in product_rows:
        stock_count = random.choice([0, 0, 3, 5, 8, 12, 20, 45])
        conn.execute(
            text("""
                INSERT INTO products (id, title, category, price, stock_count, aisle)
                VALUES (:id, :title, :category, :price, :stock_count, :aisle)
            """),
            {
                "id": product_id,
                "title": title,
                "category": category,
                "price": price,
                "stock_count": stock_count,
                "aisle": aisle,
            },
        )

print(f"Seeded {len(product_rows)} products")

Seeded 30 products


In [5]:
mock_orders = [
    ("order_001", "product_001", 2, "delivered", datetime.now() - timedelta(days=5)),
    (
        "order_002",
        "product_011",
        1,
        "out_for_delivery",
        datetime.now() - timedelta(hours=3),
    ),
    ("order_003", "product_026", 1, "placed", datetime.now() - timedelta(minutes=10)),
    ("order_004", "product_004", 3, "preparing", datetime.now() - timedelta(hours=1)),
]

with engine.begin() as conn:
    for order_id, product_id, qty, status, created_at in mock_orders:
        conn.execute(
            text("""
                INSERT INTO orders (order_id, product_id, quantity, status, created_at)
                VALUES (:order_id, :product_id, :quantity, :status, :created_at)
            """),
            {
                "order_id": order_id,
                "product_id": product_id,
                "quantity": qty,
                "status": status,
                "created_at": created_at,
            },
        )

print(f"Seeded {len(mock_orders)} mock orders")

Seeded 4 mock orders


In [6]:
from langchain_core.tools import tool


@tool
def check_stock(product_name: str) -> str:
    """Check current stock availability for a product by name (partial match supported).
    Use this when a customer asks if an item is in stock or available."""
    words = product_name.strip().split()
    if not words:
        return "Please specify a product name."

    conditions = " AND ".join([f"title ILIKE :word{i}" for i in range(len(words))])
    params = {f"word{i}": f"%{w}%" for i, w in enumerate(words)}

    with engine.connect() as conn:
        result = conn.execute(
            text(
                f"SELECT title, stock_count, price, aisle FROM products WHERE {conditions} LIMIT 3"
            ),
            params,
        ).fetchall()

    if not result:
        return f"No product found matching '{product_name}'."

    lines = []
    for title, stock, price, aisle in result:
        status = f"{stock} in stock" if stock > 0 else "OUT OF STOCK"
        lines.append(f"{title}: {status}, ₹{price}, located at {aisle}")
    return "\n".join(lines)

Wireless Bluetooth Earbuds: OUT OF STOCK, ₹1999.00, located at aisle 22


In [7]:
@tool
def place_order(product_name: str, quantity: int) -> str:
    """Place a new order for a product by name and quantity.
    Checks stock availability before confirming the order."""
    with engine.connect() as conn:
        product = conn.execute(
            text(
                "SELECT id, title, stock_count FROM products WHERE title ILIKE :pattern LIMIT 1"
            ),
            {"pattern": f"%{product_name}%"},
        ).fetchone()

    if not product:
        return f"No product found matching '{product_name}'."

    product_id, title, stock = product
    if stock < quantity:
        return f"Cannot place order — only {stock} units of '{title}' available, requested {quantity}."

    order_id = f"order_{uuid.uuid4().hex[:8]}"
    with engine.begin() as conn:
        conn.execute(
            text("""
                INSERT INTO orders (order_id, product_id, quantity, status)
                VALUES (:order_id, :product_id, :quantity, 'placed')
            """),
            {"order_id": order_id, "product_id": product_id, "quantity": quantity},
        )
        conn.execute(
            text("UPDATE products SET stock_count = stock_count - :qty WHERE id = :id"),
            {"qty": quantity, "id": product_id},
        )

    return f"Order placed! Order ID: {order_id} — {quantity}x {title}."

In [8]:
@tool
def track_order(order_id: str) -> str:
    """Look up the current status of an existing order by its order ID."""
    with engine.connect() as conn:
        result = conn.execute(
            text("""
                SELECT o.status, o.quantity, p.title, o.created_at
                FROM orders o JOIN products p ON o.product_id = p.id
                WHERE o.order_id = :order_id
            """),
            {"order_id": order_id},
        ).fetchone()

    if not result:
        return f"No order found with ID '{order_id}'."

    status, quantity, title, created_at = result
    return f"Order {order_id}: {quantity}x {title}, status = '{status}', placed on {created_at.strftime('%Y-%m-%d %H:%M')}."

In [9]:
@tool
def cancel_order(order_id: str) -> str:
    """Cancel an existing order by its order ID. Only orders not yet out for delivery can be cancelled."""
    with engine.connect() as conn:
        result = conn.execute(
            text(
                "SELECT status, product_id, quantity FROM orders WHERE order_id = :order_id"
            ),
            {"order_id": order_id},
        ).fetchone()

    if not result:
        return f"No order found with ID '{order_id}'."

    status, product_id, quantity = result
    if status in ("out_for_delivery", "delivered", "cancelled"):
        return f"Order {order_id} cannot be cancelled — current status is '{status}'."

    with engine.begin() as conn:
        conn.execute(
            text("UPDATE orders SET status = 'cancelled' WHERE order_id = :order_id"),
            {"order_id": order_id},
        )
        conn.execute(
            text("UPDATE products SET stock_count = stock_count + :qty WHERE id = :id"),
            {"qty": quantity, "id": product_id},
        )

    return f"Order {order_id} has been cancelled and refunded."

In [11]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

fulfillment_llm = ChatGroq(api_key=GROQ_API_KEY, model="openai/gpt-oss-120b")

fulfillment_agent = create_agent(
    model=fulfillment_llm,
    tools=[check_stock, place_order, track_order, cancel_order],
    system_prompt="""
You are RetailMesh's fulfillment assistant. You help customers check product
availability, place orders, track existing orders, and cancel orders. Always use the
provided tools to get real data — never guess stock levels or order statuses. Be concise
and friendly in your responses.
""",
)

In [12]:
fulfillment_test_queries = [
    "Is the wireless earbuds in stock?",
    "I want to order 2 loaves of sourdough bread",
    "What's the status of order_002?",
    "Cancel order_003",
    "Do you have the smart TV available?",
]

for q in fulfillment_test_queries:
    result = fulfillment_agent.invoke({"messages": [("user", q)]})
    final_message = result["messages"][-1].content
    print(f"\nQ: {q}")
    print(f"A: {final_message}")


Q: Is the wireless earbuds in stock?
A: The wireless Bluetooth earbuds are currently **out of stock**. Let me know if you’d like to be notified when they’re back, or if you’d prefer a similar alternative!

Q: I want to order 2 loaves of sourdough bread
A: I’m sorry, but I couldn’t find a sourdough bread item in our catalog. Could you let me know if the product has a slightly different name (e.g., “Sourdough Loaf”) or if there’s another item you’d like instead?

Q: What's the status of order_002?
A: Your order **order_002** (1 × Wireless Bluetooth Earbuds) is currently **out for delivery**. It was placed on 2026‑08‑20 at 10:11. Let me know if there’s anything else I can help with!

Q: Cancel order_003
A: Your order **order_003** has been successfully cancelled and a full refund has been processed. If you have any other requests or need further assistance, just let me know!

Q: Do you have the smart TV available?
A: I’m sorry, I don’t see any “smart TV” in our catalog. Could you let me 